In [ ]:
!pip install datasets pandas regex


# Load Hugging Face dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "charles828/vertex-ai-customer-support-training-dataset"
)
df = dataset["train"].to_pandas()


# Remove unwanted columns

In [ ]:
cols_to_drop = [c for c in ["flags", "text"] if c in df.columns]
df = df.drop(columns=cols_to_drop)


# Rename columns

In [ ]:
df = df.rename(columns={
    "instruction": "Customer_text",
    "response": "Agent_response"
})


In [ ]:
df.head()

,Customer_text,category,intent,Agent_response
0,I need support modifying the address,SHIPPING,change_shipping_address,I apologize for the inconvenience caused by th...
1,I want to locate my invoices from {{Person Nam...,INVOICE,check_invoice,I appreciate your desire to locate your invoic...
2,help me entering the secondary delivery address,SHIPPING,set_up_shipping_address,I can definitely assist you with entering your...
3,is there a way to submit feedback for your com...,FEEDBACK,review,Thank you for expressing your interest in prov...
4,can I leave feedback about a servic?,FEEDBACK,review,Positively! We value and appreciate your feedb...


## Normalize customer text

In [ ]:
import regex as re

def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

df["Customer_text_norm"] = df["Customer_text"].astype(str).apply(normalize_text)


In [ ]:
df = df.drop_duplicates(
    subset=["Customer_text_norm", "Agent_response"]
)

## Handle same instruction with multiple responses

In [ ]:
df["response_len"] = df["Agent_response"].str.split().str.len()

df = (
    df.sort_values("response_len", ascending=False)
      .drop_duplicates(subset=["Customer_text_norm"], keep="first")
)


In [ ]:
df = df.drop(columns=["Customer_text_norm", "response_len"])

## Remove responses that are too short (< 20 words)

In [ ]:
df = df[df["Agent_response"].str.split().str.len() >= 20]


## Remove placeholders

In [ ]:
PLACEHOLDER_PATTERN = r"(ORDER_ID|XXXX|YYYY|<.*?>|\[\w+\])"

df = df[~df["Agent_response"].str.contains(
    PLACEHOLDER_PATTERN,
    case=False,
    regex=True
)]


/tmp/ipython-input-2619377439.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df["Agent_response"].str.contains(


# Remove policy-related responses

In [ ]:
POLICY_PHRASES = [
    "according to our policy",
    "company policy",
    "we guarantee",
    "terms and conditions",
    "we reserve the right",
    "policy states"
]

pattern = "|".join(POLICY_PHRASES)

df = df[~df["Agent_response"].str.lower().str.contains(pattern)]


# Tone normalization

In [ ]:
BAD_TONE_PHRASES = [
    "will be resolved",
    "this will fix",
    "guaranteed",
    "no further action required"
]

pattern = "|".join(BAD_TONE_PHRASES)

df = df[~df["Agent_response"].str.lower().str.contains(pattern)]


# Add synthetic escalation data

In [ ]:
synthetic_rows = [
    {
        "Customer_text": "The invoice isnt located, I already explained this three times and nothing works",
        "Agent_response": (
            "I understand this has been frustrating, and I don’t want to keep you repeating yourself. "
            "At this point, the best next step is to involve a human support agent who can review this in detail."
        ),
        "category": "ESCALATION",
        "intent": "check_invoice"
    },
    {
        "Customer_text": "I want a refund, I want to speak to a human",
        "Agent_response": (
            "I understand your request. I’ll escalate this conversation to a human support agent "
            "who can assist you further."
        ),
        "category": "ESCALATION",
        "intent": "refund_request"
    },  {
        "Customer_text": "I am going to sue!!",
        "Agent_response": (
            "I understand your frustration, here we deaply care about your satisfation, please wait while i escalate this conversation to a human support agent "
            "who can assist you further."
        ),
        "category": "ESCALATION",
        "intent": "legal_action"
    },
     {
        "Customer_text": "i am very angry and nothing in this app is working",
        "Agent_response": (
            "I'm sorry to hear that you're experiencing difficulties. How can we assist you today to resolve the issue? If you would like, I can also transfer you to a human expert for more personalized support. Would you like me to do that?"
        ),
        "category": "ESCALATION",
        "intent": "App_issue"
    }, {
        "Customer_text": "so the service wasnt as promised ",
        "Agent_response": (
            "We regret that your experience did not meet your expectations. Your needs are important to us, and we appreciate you bringing this to our attention. I will refer you to a human agent right now"
        ),
        "category": "ESCALATION",
        "intent": "Org_fault"
    },
    {
        "Customer_text": "i diagree with everything you said so far, the shipping is all wrong",
        "Agent_response": (
            "I’m here to assist you with any concerns related to our services. Could you please provide more details about the specific issues you are facing? I’m here to assist you with any concerns related to our services. Could you please provide more details about the specific issues you are facing? "
        ),
        "category": "ESCALATION",
        "intent": "shipping_issue"
    },
    {
        "Customer_text": "can you tell me more about global warming and it's impact on plastic production",
        "Agent_response": (
            "I'm sorry, but I couldn't find any information related to your question about global warming and plastic production. Could you please rephrase your question or let me know if you'd like help with something else?"
        ),
        "category": "Other",
        "intent": "Other"
    },
    {
        "Customer_text": "Hi! My order was supposed to arrive yesterday, but the tracking hasn’t updated in 3 days. Can you check why it’s delayed?",
        "Agent_response": (
            "Thanks for reaching out! I can see your order for the laptop. Unfortunately, I don’t have access to detailed shipping updates or carrier systems to check on delays. I’ll connect you to our Human Agent in a minute, please wait"
                    ),
        "category": "ESCALATION",
        "intent": "Shipping"
    }, {
        "Customer_text": "Hi, why did my account get suspended",
        "Agent_response": (
            "I'm sorry to hear about your account suspension. I'll connect you with one of our human agents who can assist you further with this issue."
        ),
        "category": "ESCALATION",
        "intent": "Account_suspended"
    },


]

import pandas as pd
df = pd.concat([df, pd.DataFrame(synthetic_rows)], ignore_index=True)


# Final checks

In [ ]:
print("Final rows:", len(df))
df.sample(5)

Final rows: 20783


,Customer_text,category,intent,Agent_response
14205,I need to swqap something of purchase {{Order ...,ORDER,change_order,Thank you for trusting to us! We understand th...
10775,can ya help me demand a rebate of money,REFUND,get_refund,"Certainly, I'm here to assist you in requestin..."
3071,I have to use the bloody {{Account Type}} prof...,ACCOUNT,switch_account,Positively! I'm here to assist you in harnessi...
2057,i do not know how to set my shipping address up,SHIPPING,set_up_shipping_address,I'm aware that you're feeling unsure about how...
12660,correcting infornation on standard account,ACCOUNT,edit_account,How wonderful it is to embark on the path of c...


In [ ]:
df.to_csv("clean_customer_support_dataset.csv", index=False)